# UCI Adult — baseline and initial model comparison

This stage fits simple candidates to establish a reference. The official test set remains reserved for final evaluation. `sex` and `race` are retained for auditing and are initially excluded from the features.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == 'en-US' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from adult_data import load_adult
from baseline_models import write_baseline_outputs
frame = load_adult(ROOT / 'data' / 'raw' / 'adult_uci')
outputs = write_baseline_outputs(frame, ROOT / 'data' / 'processed' / 'baseline')
outputs

## Split contract

The official training split is divided into fitting and calibration/validation subsets. `adult.test` is not used for selection, threshold tuning, or the initial comparison.

In [ ]:
import json
json.loads(Path(outputs['contract']).read_text(encoding='utf-8'))

## Model comparison

Guiding question: which candidate provides the best starting point considering discrimination, probability quality, and balance across metrics?

The metrics below are calculated on calibration/validation, using the provisional 0.5 threshold only for classification metrics. ROC-AUC and PR-AUC evaluate ranking; Brier score evaluates probability quality. Final selection has not been made.

In [ ]:
import pandas as pd
comparison = pd.read_csv(outputs['comparison'])
comparison

### Insights:

`DummyClassifier` establishes the comparison floor. A useful model must exceed this floor in discrimination and probability quality. `HistGradientBoosting` reached ROC-AUC 0.928213 and PR-AUC 0.833058 on calibration/validation, a strong result at this stage and clearly above logistic regression (ROC-AUC 0.906815) and Random Forest (0.919201).

This 0.928213 is encouraging, but it should be interpreted as evidence of strong ranking ability on this calibration split, not as guaranteed final performance. The model still needs calibration, stability, sex and race subgroup evaluation, and final evaluation on the reserved official test set.

No single metric defines the selected model, and this comparison cannot yet establish which candidate is acceptable across groups. Fairness auditing will follow for advancing candidates without consuming the official test set early.

## Next questions

1. How do candidates behave by sex, race, and their intersection?
2. Do the probabilities require calibration?
3. Which model offers the best compromise between performance, stability, and interpretability?
4. Which threshold can be justified without using the test set?

These questions will be addressed in the calibration and audit stages.